# NexusTrade — DCA & Break-Even Calculator
## Analyze Multiple Entries and Calculate Average Cost

⚠️ **WARNING**: Averaging down is risky. Only DCA if the original thesis remains valid!

In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime

class DCACalculator:
    def __init__(self):
        self.entries = []
    
    def add_entry(self, date, shares, price):
        """Add a buy entry"""
        cost = shares * price
        self.entries.append({
            'date': date,
            'shares': shares,
            'price': price,
            'cost': cost
        })
    
    def calculate_summary(self, current_price):
        """Calculate position summary"""
        if not self.entries:
            return {"error": "No entries added"}
        
        df = pd.DataFrame(self.entries)
        
        total_shares = df['shares'].sum()
        total_cost = df['cost'].sum()
        avg_price = total_cost / total_shares
        
        current_value = total_shares * current_price
        unrealized_pnl = current_value - total_cost
        unrealized_pct = (unrealized_pnl / total_cost) * 100
        
        break_even = avg_price
        to_break_even_pct = ((break_even - current_price) / current_price) * 100
        
        return {
            'entries_df': df,
            'total_shares': int(total_shares),
            'total_cost': round(total_cost, 2),
            'avg_cost': round(avg_price, 4),
            'current_price': current_price,
            'current_value': round(current_value, 2),
            'unrealized_pnl': round(unrealized_pnl, 2),
            'unrealized_pct': round(unrealized_pct, 2),
            'break_even': round(break_even, 4),
            'to_break_even_pct': round(to_break_even_pct, 2)
        }
    
    def simulate_new_entry(self, new_shares, new_price, current_price):
        """Simulate adding shares without committing"""
        current = self.calculate_summary(current_price)
        
        new_total_shares = current['total_shares'] + new_shares
        new_total_cost = current['total_cost'] + (new_shares * new_price)
        new_avg_cost = new_total_cost / new_total_shares
        
        return {
            'old_avg': current['avg_cost'],
            'new_avg': round(new_avg_cost, 4),
            'avg_reduction': round(current['avg_cost'] - new_avg_cost, 4),
            'avg_reduction_pct': round(((current['avg_cost'] - new_avg_cost) / current['avg_cost']) * 100, 2),
            'new_total_shares': new_total_shares,
            'new_total_cost': round(new_total_cost, 2),
            'additional_capital': new_shares * new_price,
            'new_unrealized_pct': round(((current_price - new_avg_cost) / new_avg_cost) * 100, 2)
        }
    
    def clear_entries(self):
        """Reset calculator"""
        self.entries = []

## Example: DVLT Multiple Entry Analysis

In [ ]:
# VOORBEELD: DVLT Position met meerdere entries
calc = DCACalculator()

# Scenario: Gekocht op verschillende prijzen
calc.add_entry("2026-04-10", 500, 1.00)
calc.add_entry("2026-04-15", 800, 0.85)
calc.add_entry("2026-04-20", 755, 0.70)

current_price = 0.7386

summary = calc.calculate_summary(current_price)

print("═══════════════════════════════════════════")
print("DCA POSITION ANALYSIS")
print("═══════════════════════════════════════════")
print("\nTRADE HISTORY:")
print(summary['entries_df'].to_string(index=False))
print("\n" + "─" * 47)
print(f"TOTAL                {summary['total_shares']:>4}   AVG: ${summary['avg_cost']:<6}  ${summary['total_cost']:>8.2f}")
print("═══════════════════════════════════════════")
print(f"\nCurrent Price:      ${summary['current_price']}")
print(f"Current Value:      ${summary['current_value']:,.2f}")
print(f"Unrealized P&L:     ${summary['unrealized_pnl']:,.2f} ({summary['unrealized_pct']:+.2f}%)")
print(f"Break-even Price:   ${summary['break_even']}")
print(f"To Break-even:      {summary['to_break_even_pct']:+.2f}%")
print("═══════════════════════════════════════════")

# Analysis
if summary['unrealized_pct'] < -20:
    print("\n🔴 WARNING: Position down >20% — consider stop loss")
elif summary['unrealized_pct'] < -10:
    print("\n⚠️  CAUTION: Position down >10% — monitor closely")
elif summary['unrealized_pct'] < 0:
    print("\n🟠 Underwater but manageable — have exit plan ready")
else:
    print("\n✅ Position in profit — consider trailing stop")

## Simulation: Should You Add More?
⚠️ Only use if thesis is still valid and you have a plan!

In [ ]:
# Simuleer nog een entry ZONDER te committen
print("\nSIMULATION: What if you add 500 shares @ $0.65?")
print("─" * 50)

sim = calc.simulate_new_entry(500, 0.65, current_price)

print(f"Old Average Cost:   ${sim['old_avg']}")
print(f"New Average Cost:   ${sim['new_avg']}")
print(f"Average Reduction:  ${sim['avg_reduction']} ({sim['avg_reduction_pct']:.2f}%)")
print(f"\nAdditional Capital: ${sim['additional_capital']:.2f}")
print(f"New Total Shares:   {sim['new_total_shares']}")
print(f"New Total Cost:     ${sim['new_total_cost']:.2f}")
print(f"New Unrealized:     {sim['new_unrealized_pct']:+.2f}%")

print("\n⚠️  CRITICAL QUESTIONS BEFORE ADDING:")
print("   1. Is the original thesis STILL valid?")
print("   2. Did fundamentals change (insider selling, dilution)?")
print("   3. Am I averaging down or doubling down on a mistake?")
print("   4. Can I afford to lose this additional capital?")
print("   5. What's my max loss tolerance on total position?")

## Interactive: Your Position Analysis

In [ ]:
# ==== ENTER YOUR ACTUAL TRADES ====
my_calc = DCACalculator()

# Add your entries here (date, shares, price)
my_calc.add_entry("2026-04-10", 500, 1.00)
my_calc.add_entry("2026-04-15", 800, 0.85)
my_calc.add_entry("2026-04-20", 755, 0.70)

MY_CURRENT_PRICE = 0.7386
# ===================================

my_summary = my_calc.calculate_summary(MY_CURRENT_PRICE)

print("\n🎯 YOUR POSITION BREAKDOWN:")
print(my_summary['entries_df'].to_string(index=False))
print(f"\nAverage Cost: ${my_summary['avg_cost']}")
print(f"Current P&L:  ${my_summary['unrealized_pnl']} ({my_summary['unrealized_pct']:+.2f}%)")
print(f"To Break-even: {my_summary['to_break_even_pct']:+.2f}% move needed")

## Scenario Analysis: Multiple Add Points

In [ ]:
# Test verschillende DCA scenario's
add_prices = [0.60, 0.65, 0.70, 0.75]
add_shares = 500

print("\nDCA SCENARIO ANALYSIS (adding 500 shares):")
print("─" * 70)
print(f"{'Add Price':<12} {'New Avg':<12} {'Reduction':<12} {'New P&L':<12} {'Capital':<12}")
print("─" * 70)

for price in add_prices:
    sim = calc.simulate_new_entry(add_shares, price, current_price)
    print(f"${price:<11.2f} ${sim['new_avg']:<11.4f} ${sim['avg_reduction']:<11.4f} "
          f"{sim['new_unrealized_pct']:>+10.2f}% ${sim['additional_capital']:<11.2f}")

print("\n💡 Key Insight:")
print("Adding at lower prices reduces average BUT increases total capital at risk.")
print("Make sure you're not throwing good money after bad!")

## Red Flag Check: Is Your DCA Strategy Broken?

In [ ]:
def dca_health_check(summary, portfolio_value):
    """Check if DCA strategy is healthy or broken"""
    position_pct = (summary['total_cost'] / portfolio_value) * 100
    
    issues = []
    
    if summary['unrealized_pct'] < -30:
        issues.append("🔴 CRITICAL: Down >30% — cut losses NOW")
    elif summary['unrealized_pct'] < -20:
        issues.append("🔴 Down >20% — re-evaluate thesis immediately")
    elif summary['unrealized_pct'] < -10:
        issues.append("⚠️  Down >10% — have exit plan ready")
    
    if position_pct > 50:
        issues.append("🔴 CRITICAL: Position >50% of portfolio — extreme overexposure")
    elif position_pct > 20:
        issues.append("🔴 Position >20% — too concentrated")
    elif position_pct > 10:
        issues.append("⚠️  Position >10% — monitor closely")
    
    if len(summary['entries_df']) > 5:
        issues.append("⚠️  More than 5 entries — may be chasing losses")
    
    # Check if adding at increasingly lower prices (knife-catching)
    prices = summary['entries_df']['price'].tolist()
    if len(prices) > 2 and all(prices[i] > prices[i+1] for i in range(len(prices)-1)):
        issues.append("🔴 Knife-catching pattern detected — each buy is lower")
    
    return issues

# Check your position health
PORTFOLIO_VALUE = 2850
health_issues = dca_health_check(summary, PORTFOLIO_VALUE)

print("\n═══════════════════════════════════════════")
print("DCA HEALTH CHECK")
print("═══════════════════════════════════════════")

if not health_issues:
    print("✅ Position appears healthy")
    print("Continue monitoring and stick to your exit plan")
else:
    print("⚠️  WARNING: Issues detected:\n")
    for issue in health_issues:
        print(f"   {issue}")
    print("\n💡 Consider de-risking or exiting this position")